In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle

In [2]:
# Load the dataste

df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocess the data
### Drop irrelevent columns

df = df.drop(['RowNumber','CustomerId','Surname'],axis = 1)


In [4]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 966.1 KB


In [6]:
# Dividing the dataset into train & test

X = df.drop('Exited',axis=1)
y = df['Exited']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)


In [ ]:
# Encoding categorical data

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# ColumnTranform expects column name not the dataset!!
preprocessor = ColumnTransformer( 
    transformers=[
        ('binary', OrdinalEncoder(categories=[['Female','Male']]), ['Gender']),
        ('ohe',OneHotEncoder(sparse_output=False),['Geography']),
        ('Scale',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)


# Transforming the columns (array!!)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# convert back to dataframe (concatinate these arrays into dataframe)

df_X_transformed = pd.DataFrame(
    X_train_transformed, 
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)


In [ ]:
df_X_transformed.head()

In [ ]:
X_train_transformed

,Gender,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
9254,1.0,1.0,0.0,0.0,0.356500,-0.655786,0.345680,-1.218471,0.808436,0.649203,0.974817,1.367670
1561,1.0,0.0,1.0,0.0,-0.203898,0.294938,-0.348369,0.696838,0.808436,0.649203,0.974817,1.661254
1670,1.0,0.0,0.0,1.0,-0.961472,-1.416365,-0.695393,0.618629,-0.916688,0.649203,-1.025834,-0.252807
6087,0.0,1.0,0.0,0.0,-0.940717,-1.131148,1.386753,0.953212,-0.916688,0.649203,-1.025834,0.915393
6669,1.0,1.0,0.0,0.0,-1.397337,1.625953,1.386753,1.057449,-0.916688,-1.540351,-1.025834,-1.059600


In [9]:
# Pickel Encoding pipeline

import pickle

with open("preprocessor.pkl","wb") as f:
    pickle.dump(preprocessor, f)

In [10]:
# Check weather it file os created or not!
import os

print(os.path.exists("preprocessor.pkl"))

True


# ANN Implementation

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [16]:
(X_train_transformed.shape[1],)

(12,)

In [18]:
# Build ANN Model

model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train_transformed.shape[1],)), ## HL1 Connected with input layer
    Dense(32,activation='relu'), # HL2 # since it is sequential it knows inputs will be connected so no need to give input_shape
    Dense(1,activation='sigmoid') # Output layer 

])

In [20]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate = 0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

In [26]:
## Compile the model 

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])


In [30]:
## Set up Tensorboard

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [31]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
# Train the model
history = model.fit(
    X_train_transformed,y_train,validation_data=(X_test_transformed,y_test), epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
## Load Tensorboard Extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit